In [5]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, fullname + ".ipynb")
        path = os.path.normpath(path)

        # Load the notebook
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # Create a module to store the imported functions and classes
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # Go through the notebook cells and only execute function or class definitions
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # Check for function or class definitions
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "GPT2_to_Lamma-2"
    names = ["precompute_rope_params", "compute_rope", "SiLU", "FeedForward", "RMSNorm", "MultiHeadAttention"]

    return import_definitions_from_notebook(fullname, names)

In [11]:
# import torch.nn as nn
imported_module = import_from_notebook()

# We need to redefine precompute_rope_params
# precompute_rope_params = getattr(imported_module, "precompute_rope_params", None)
compute_rope = getattr(imported_module, "compute_rope", None)
SiLU = getattr(imported_module, "SiLU", None)
FeedForward = getattr(imported_module, "FeedForward", None)
RMSNorm = getattr(imported_module, "RMSNorm", None)

# MultiHeadAttention only for comparison purposes
MultiHeadAttention = getattr(imported_module, "MultiHeadAttention", None)

In [13]:
import torch

 Llama 3 now supports up to 8,192 tokens, twice as many as Llama 2 (4,096)
The base value for the so-called RoPE 
 (see equation below) was increased from 10,000 (Llama 2) to 500,000 (Llama 3)

Perfect — this is **the most important (and most confusing) part of RoPE long-context scaling**. Let’s slow it down and answer **two core questions**:

> **WHY do we do this?**
> **HOW does this actually work inside attention?**

I’ll explain **intuition first**, then **line-by-line mechanics**, then **what would break without it**.

---

# 1️⃣ Why do we need `freq_config` at all?

### Problem: RoPE breaks when context grows

RoPE encodes position using **rotations**:
[
\theta_{pos,i} = pos \cdot \omega_i
]

When you go **far beyond the context length used in training**:

* Angles become **too large**
* Rotations **wrap around many times**
* Attention can’t distinguish far-away positions
* Model becomes unstable

👉 **Low-frequency dimensions break first**
(those meant to represent *long-range structure*)

---

# 2️⃣ The core idea (one sentence)

> **Slow down the rotations that encode long-range information, but keep short-range rotations intact.**

That’s exactly what this code does.

---

# 3️⃣ Frequency → wavelength (key mental shift)

```python
wavelen = 2 * torch.pi / inv_freq
```

Why convert?

| Concept    | Meaning                           |
| ---------- | --------------------------------- |
| `inv_freq` | How fast angle rotates            |
| `wavelen`  | How many tokens per full rotation |

🔑 **Large wavelength = long-range dimension**

---

# 4️⃣ Split frequencies into bands

```python
low_freq_wavelen  = original_ctx / low_freq_factor
high_freq_wavelen = original_ctx / high_freq_factor
```

This creates **three zones**:

```
long-range     medium-range     short-range
───────────|──────────────|──────────────
 wavelen > low   in between        wavelen < high
```

| Band      | Role             | Action         |
| --------- | ---------------- | -------------- |
| Low freq  | Global structure | Strong scaling |
| Medium    | Mixed info       | Smooth scaling |
| High freq | Local detail     | No scaling     |

---

# 5️⃣ Scale LOW frequencies (most important part)

```python
inv_freq_llama = torch.where(
    wavelen > low_freq_wavelen,
    inv_freq / factor,
    inv_freq
)
```

### What this does mathematically

Original angle:

theta = pos * omega


Scaled:

theta = pos * (omega/factor)


So rotations happen **factor× slower**

---

### Why only low frequencies?

Because:

* Low frequencies rotate **slowly**
* They represent **long-range dependencies**
* They break first at large context lengths

This is the **NTK scaling trick** used in LLaMA-2.

---

# 6️⃣ Why smoothing is necessary (VERY important)

If we suddenly jump from:

```
inv_freq → inv_freq / factor
```

We introduce:

* Discontinuities
* Phase mismatch between dimensions
* Attention instability

❌ **Hard boundaries break models**

---

# 7️⃣ Smooth transition for medium frequencies

```python
smooth_factor = (
    original_ctx / wavelen - low_freq_factor
) / (
    high_freq_factor - low_freq_factor
)
```

This computes a **blend weight** ∈ [0, 1]:

| Value | Meaning      |
| ----- | ------------ |
| 0     | fully scaled |
| 1     | no scaling   |

---

### Interpolation

```python
smoothed_inv_freq =
    (1 - smooth_factor) * (inv_freq / factor)
  + smooth_factor * inv_freq
```

📌 This line is doing **linear interpolation** between:

* scaled frequency
* original frequency

---

# 8️⃣ Apply smoothing only to medium frequencies

```python
is_medium_freq = (
    wavelen <= low_freq_wavelen
) & (
    wavelen >= high_freq_wavelen
)
```

```python
inv_freq_llama = torch.where(
    is_medium_freq,
    smoothed_inv_freq,
    inv_freq_llama
)
```

So final behavior:

| Frequency band | Result                |
| -------------- | --------------------- |
| Low freq       | `inv_freq / factor`   |
| Medium         | smoothly interpolated |
| High           | unchanged             |

---

# 9️⃣ How this helps attention (mechanics)

Attention uses dot products:

[
(Q e^{i\theta_q}) \cdot (K e^{i\theta_k})
]

Which becomes:
[
Q \cdot K \cdot \cos(\theta_q - \theta_k)
]

By slowing rotations:

* `θ_q - θ_k` remains meaningful
* Relative positions still correlate
* Long-range tokens don’t alias

---

# 🔥 What would happen without this?

| Without scaling    | With scaling                 |
| ------------------ | ---------------------------- |
| Phase wrapping     | Stable rotations             |
| Loss spikes        | Smooth attention             |
| Context collapse   | Long-context support         |
| Position confusion | Relative structure preserved |




In [14]:
def precompute_rope_params(head_dim, theta_base=10_000, context_length=4096, freq_config=None):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))

    # Frequency adjustments
    # //////////// NEW ///////
    if freq_config is not None:
        low_freq_wavelen = freq_config["original_context_length"] / freq_config["low_freq_factor"]
        high_freq_wavelen = freq_config["original_context_length"] / freq_config["high_freq_factor"]

        wavelen = 2 * torch.pi / inv_freq

        inv_freq_llama = torch.where(
            wavelen > low_freq_wavelen, inv_freq / freq_config["factor"], inv_freq
        )

        smooth_factor = (freq_config["original_context_length"] / wavelen - freq_config["low_freq_factor"]) / (
            freq_config["high_freq_factor"] - freq_config["low_freq_factor"]
        )

        smoothed_inv_freq = (
            (1 - smooth_factor) * (inv_freq / freq_config["factor"]) + smooth_factor * inv_freq
        )

        is_medium_freq = (wavelen <= low_freq_wavelen) & (wavelen >= high_freq_wavelen)
        inv_freq_llama = torch.where(is_medium_freq, smoothed_inv_freq, inv_freq_llama)
        inv_freq = inv_freq_llama
    # ///////////////////// ////////////


    # Generate position indices
    positions = torch.arange(context_length)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


In [12]:
llama_2_context_len = 4096
llama_3_context_len = 8192

llama_2_theta_base = 10_000
llama_3_theta_base = 500_000

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(
            self, d_in, d_out, num_heads,
            num_kv_groups,       # NEW
            dtype=None
        ):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"  # NEW

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        ############################# NEW  #############################
        # self.W_key = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        # self.W_value = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=False, dtype=dtype)
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups
        ################################################################

        self.W_query = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype)


    def forward(self, x, mask=None, cos=None, sin=None):
        ##################### NEW  #####################
        # The forward method now accepts `mask` instead of accessing it via self.mask.
        # Also, we now have cos and sin as input for RoPE
        ################################################    
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)  # Shape: (b, num_tokens, d_out)
        keys = self.W_key(x)  # Shape: (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)  # Shape: (b, num_tokens, num_kv_groups * head_dim)

        # Reshape queries, keys, and values
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        ##################### NEW  #####################
        # keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        # values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim)
        ################################################

        # Transpose keys, values, and queries
        keys = keys.transpose(1, 2)  # Shape: (b, num_kv_groups, num_tokens, head_dim)
        values = values.transpose(1, 2)  # Shape: (b, num_kv_groups, num_tokens, head_dim)
        queries = queries.transpose(1, 2)  # Shape: (b, num_heads, num_tokens, head_dim)

        ##################### NEW #####################
        # Apply RoPE
        if cos is not None:
            keys = compute_rope(keys, cos, sin)
            queries = compute_rope(queries, cos, sin)
        ################################################

        ##################### NEW  #####################
        # Expand keys and values to match the number of heads
        # Shape: (b, num_heads, num_tokens, head_dim)

        keys = keys.repeat_interleave(self.group_size, dim=1)  # Shape: (b, num_heads, num_tokens, head_dim)
        values = values.repeat_interleave(self.group_size, dim=1)  # Shape: (b, num_heads, num_tokens, head_dim)
        # For example, before repeat_interleave along dim=1 (query groups):
        #   [K1, K2]
        # After repeat_interleave (each query group is repeated group_size times):
        #   [K1, K1, K2, K2]
        # If we used regular repeat instead of repeat_interleave, we'd get:
        #   [K1, K2, K1, K2]
        ################################################

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        # Shape: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        ##################### NEW #####################
        # Create mask on the fly
        if mask is None:
            mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)
        ################################################
    
        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        assert keys.shape[-1] == self.head_dim

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec